# 01 — 数据概览与清洗

解析问卷平台导出的汇总 Excel 和逐名文本版答卷矩阵，并确认真实样本量、题型和可分析的数据粒度。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1. 解析问卷汇总导出

In [2]:
from src.data_loader import load_participant_survey, parse_survey_export
from src.preprocessing import attach_slider_constructs

parsed = parse_survey_export(ROOT / "data" / "问卷原始数据.xlsx")
questions = parsed["questions"]
options = parsed["options"]
sliders = attach_slider_constructs(parsed["sliders"])
participants = load_participant_survey(ROOT / "data" / "processed" / "问卷数据_文本版.xlsx")

print(f"Questions: {len(questions)}")
print(f"Participant-level core rows: {len(participants)}")
display(questions.head(12))

Questions: 29
Participant-level core rows: 32


,q_num,question,type,valid_n,total_score,mean,source_row
0,1,您目前是否为在校大学生？,单选题,34.0,NaN,NaN,1
1,2,您的性别认同是？,单选题,33.0,NaN,NaN,7
2,3,您的年龄是？,单选题,32.0,NaN,NaN,13
3,4,过去一个月，您平均每周进行运动的频率大约是？,单选题,32.0,NaN,NaN,21
4,5,您是否使用过学校健身房？,单选题,32.0,NaN,NaN,30
5,6,您通常在什么时间段去健身房？（可多选）,多选题,32.0,NaN,NaN,36
6,7,您最常使用健身房哪些区域？（可多选）,多选题,32.0,NaN,NaN,45
7,8,您是否曾系统接触过力量训练？,单选题,32.0,NaN,NaN,55
8,9,您当前最主要的运动目标是？（最多选2项）,多选题,32.0,NaN,NaN,63
9,10,自由重量区的整体布局让我觉得不容易接近。,滑动条,32.0,102.2,3.19,75


## 2. 核心有效样本

第 1、2 题为筛选题；第 3 题开始的核心样本为符合纳入条件的女性在校大学生。

In [3]:
screening = questions[questions["q_num"].isin([1, 2, 3])]
display(screening[["q_num", "question", "valid_n"]])
print("Core analytic N =", int(questions.loc[questions["q_num"].eq(3), "valid_n"].iloc[0]))

,q_num,question,valid_n
0,1,您目前是否为在校大学生？,34.0
1,2,您的性别认同是？,33.0
2,3,您的年龄是？,32.0


Core analytic N = 32


## 3. 题项频数与缺失标记

多选题导出中少数选项为空白，不能可靠地区分为 0 还是平台导出缺失；脚本保留 `count_status` 以免误读。单选题若只有一个选项缺失且总数可由有效样本推出，则标记为 `inferred_from_valid_n`。

In [4]:
display(options[options["q_num"].isin([16, 17, 18])][["q_num", "option", "count", "pct", "count_status"]])

,q_num,option,count,pct,count_status
68,16,从未,5.0,15.63,reported
69,16,很少,5.0,15.63,reported
70,16,有时,10.0,31.25,reported
71,16,经常,6.0,18.75,reported
72,16,总是,6.0,18.75,inferred_from_valid_n
73,17,男性太多,19.0,59.38,reported
74,17,害怕被注视,NaN,NaN,missing
75,17,害怕动作出错,9.0,28.13,reported
76,17,不会使用器械,19.0,59.38,reported
77,17,空间太拥挤,NaN,NaN,missing


## 4. 保存结构化数据

In [5]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
questions.to_csv(out_dir / "survey_questions.csv", index=False, encoding="utf-8-sig")
options.to_csv(out_dir / "survey_options.csv", index=False, encoding="utf-8-sig")
sliders.to_csv(out_dir / "survey_slider_items.csv", index=False, encoding="utf-8-sig")
print(out_dir)

E:\Project\1808论文\data\processed
